# Snowflake REST API - JWT 토큰 생성하기

Snowflake REST API 인증을 위한 JWT 토큰을 생성하는 과정입니다.

**전체 흐름:**
1. RSA 키 쌍 생성 (공개키 + 개인키)
2. 공개키를 Snowflake 사용자에게 등록
3. 공개키 fingerprint(SHA-256) 계산
4. JWT 토큰 생성 (iss, sub, iat, exp)
5. REST API 호출 시 Authorization 헤더에 토큰 포함

## Step 1: RSA 키 쌍 생성

2048비트 RSA 키 쌍을 생성하고, PEM 형식으로 출력합니다.

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

# RSA 키 쌍 생성 (2048비트)
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
    backend=default_backend()
)

# 개인키 PEM 형식 (암호화 없이)
private_key_pem = private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
).decode('utf-8')

# 공개키 PEM 형식
public_key_pem = private_key.public_key().public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
).decode('utf-8')

print("=== 개인키 (rsa_key.p8) ===")
print(private_key_pem[:100] + "...")
print()
print("=== 공개키 (rsa_key.pub) ===")
print(public_key_pem)

## Step 2: 공개키를 Snowflake 사용자에 등록

아래 SQL을 실행하여 공개키를 사용자에게 할당합니다.  
(BEGIN/END 사이의 내용만 복사하여 사용)

In [ ]:
# 공개키에서 헤더/푸터 제거 (Snowflake가 요구하는 형식)
public_key_raw = public_key_pem.replace("-----BEGIN PUBLIC KEY-----", "") \
                               .replace("-----END PUBLIC KEY-----", "") \
                               .replace("\n", "")

print("아래 SQL을 실행하여 공개키를 등록하세요:")
print()
print(f"ALTER USER ADMIN SET RSA_PUBLIC_KEY='{public_key_raw}';")
print()
print("등록 확인:")
print("DESCRIBE USER ADMIN;  -- RSA_PUBLIC_KEY_FP 값을 확인")

In [ ]:
ALTER USER ADMIN SET RSA_PUBLIC_KEY='MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAkvMK+f7D3osZX7I7yT24628U0CKedo7MHY7yVA+WDde7h3ZSy5AQ4eziLMUnX1EWO3iwwfc5Q2ceexTuIgKz+xyybU5YICrzeMqwsGpfEY9Dd4sEueVWzAiseZ6kuzvw1dAA0LMQq2ir2EleeLBGqydu4cIDvID4+u2LTttp4KmO1tfW9TdG2LITTUmRDRu41m5pLeJdQm6m8d5nPr3S1Bj9Rn9ttKnAGpEm7LMMmEvobUbcgvX2UlN98t0mhPPvj/XpTL295DAgqRP7CvaDTX7D246Jaf2vOJylex2Gh19kG66vPookVXqrYl6A8LXKW7LEtlj6og3rabEwl75RdQIDAQAB';

-- 등록 확인:
DESCRIBE USER ADMIN;  -- RSA_PUBLIC_KEY_FP 값을 확인

## Step 3: 공개키 Fingerprint 계산

JWT의 `iss` 클레임에 필요한 SHA-256 fingerprint를 계산합니다.

In [ ]:
import hashlib
import base64

# 공개키의 DER 인코딩 → SHA-256 해시 → Base64
public_key_der = private_key.public_key().public_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
)

sha256_hash = hashlib.sha256(public_key_der).digest()
fingerprint = base64.b64encode(sha256_hash).decode('utf-8')
public_key_fp = f"SHA256:{fingerprint}"

print(f"Public Key Fingerprint: {public_key_fp}")

## Step 4: JWT 토큰 생성

JWT payload 구성:
- **iss**: `<ACCOUNT>.<USER>.SHA256:<fingerprint>`
- **sub**: `<ACCOUNT>.<USER>`
- **iat**: 발급 시각 (UTC)
- **exp**: 만료 시각 (최대 1시간)

> 주의: account_identifier와 user는 반드시 **대문자**여야 합니다.

In [ ]:
from datetime import datetime, timezone, timedelta
import jwt

# 계정 정보 설정 (대문자 필수)
ACCOUNT = "FZNFPOY-MB25649"   # 본인의 account_identifier (org-account 형식)
USER = "ADMIN"                 # Snowflake 사용자명

qualified_username = f"{ACCOUNT}.{USER}"

# JWT Payload
now = datetime.now(timezone.utc)
payload = {
    "iss": f"{qualified_username}.{public_key_fp}",
    "sub": qualified_username,
    "iat": now,
    "exp": now + timedelta(minutes=59),  # 최대 1시간
}

# RS256으로 서명
token = jwt.encode(payload, private_key, algorithm="RS256")

print("=" * 60)
print("JWT Payload:")
print(f"  iss: {payload['iss']}")
print(f"  sub: {payload['sub']}")
print(f"  iat: {payload['iat']}")
print(f"  exp: {payload['exp']}")
print("=" * 60)
print()
print("생성된 JWT 토큰:")
print(token)

## Step 5: REST API 호출 예시

생성된 JWT 토큰을 사용하여 Snowflake REST API를 호출하는 방법입니다.

In [ ]:
import requests

# REST API 엔드포인트
account_url = "fznfpoy-mb25649"  # 소문자 가능
base_url = f"https://{account_url}.snowflakecomputing.com"

# 헤더 설정
headers = {
    "Authorization": f"Bearer {token}",
    "X-Snowflake-Authorization-Token-Type": "KEYPAIR_JWT",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

print("REST API 호출 예시:")
print(f"  URL: {base_url}/api/v2/databases")
print(f"  Headers:")
for k, v in headers.items():
    if k == "Authorization":
        #print(f"    {k}: Bearer {token[:50]}...")
        print(f"    {k}: Bearer {token}")
    else:
        print(f"    {k}: {v}")
print()
print("# 실제 호출 (주석 해제하여 사용):")
print("# response = requests.get(f'{base_url}/api/v2/databases', headers=headers)")
print("# print(response.status_code)")
print("# print(response.json())")

In [ ]:
%%sql -r alter_user_result
ALTER USER ADMIN SET RSA_PUBLIC_KEY='{{public_key_raw}}';

In [ ]:
%%sql -r user_desc
DESCRIBE USER ADMIN;

## Step 6: REST API 테스트 (curl / Postman)

노트북 내부에서는 외부 네트워크 호출이 차단됩니다.  
아래 출력되는 **curl 명령어**를 터미널에서 실행하거나, **Postman**에서 토큰을 붙여넣어 테스트하세요.

In [ ]:
# 노트북 내부에서는 외부 네트워크 호출이 제한됩니다.
# 아래 curl 명령어를 터미널 또는 Postman에서 사용하세요.

account_url = "fznfpoy-mb25649"
base_url = f"https://{account_url}.snowflakecomputing.com"

print("=" * 60)
print("GET - 데이터베이스 목록 조회")
print("=" * 60)
print(f"""curl -X GET '{base_url}/api/v2/databases' \\
  -H 'Authorization: Bearer {token}' \\
  -H 'X-Snowflake-Authorization-Token-Type: KEYPAIR_JWT' \\
  -H 'Content-Type: application/json' \\
  -H 'Accept: application/json'""")

print()
print("=" * 60)
print("POST - 데이터베이스 생성")
print("=" * 60)
print(f"""curl -X POST '{base_url}/api/v2/databases?createMode=ifNotExists' \\
  -H 'Authorization: Bearer {token}' \\
  -H 'X-Snowflake-Authorization-Token-Type: KEYPAIR_JWT' \\
  -H 'Content-Type: application/json' \\
  -H 'Accept: application/json' \\
  -d '{{"name": "TEST_API_DB"}}'""")

print()
print("=" * 60)
print("DELETE - 데이터베이스 삭제")
print("=" * 60)
print(f"""curl -X DELETE '{base_url}/api/v2/databases/TEST_API_DB?ifExists=true' \\
  -H 'Authorization: Bearer {token}' \\
  -H 'X-Snowflake-Authorization-Token-Type: KEYPAIR_JWT' \\
  -H 'Content-Type: application/json' \\
  -H 'Accept: application/json'""")